# Illustration of information leakage

In [1]:
# For CV
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

# For preprocessing/feature selection
from sklearn.feature_selection import SelectKBest, f_regression, f_classif

# Models
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

### Regression setup

Below, we fit a regression model to a dataset `X` consisting of pure noise (random data, unrelated to the target `y`).

Knowing this, no model should be able to predict `y` from `X` better than random guessing, resulting in an $R^2$ score close to or smaller than 0.

Since that dataset is really large, we perform a preprocessing step, in which we select the "most promising" features.

If we implement cross-validation incorrectly, we will see that the model appears to perform very well, even though this is impossible.

In [2]:
# Configurable params
k_best = 50
estimator = LinearRegression()
rng = np.random # .RandomState(0) # for reproducibility

# Data setup (pure noise!)
X = rng.normal(size=(200, 10000))   # random features
y = rng.normal(size=200)          # independent random target

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=0)

In [3]:
# BAD: feature selection fit on full data (leakage!)
X_selected = SelectKBest(f_regression, k=k_best).fit_transform(X, y)
bad_scores = cross_val_score(estimator, X_selected, y, cv=cv, scoring="r2")

In [4]:
# GOOD: feature selection inside a Pipeline
pipe = Pipeline([
    ("select", SelectKBest(f_regression, k=k_best)),
    ("est", estimator),
])
good_scores = cross_val_score(pipe, X, y, cv=cv, scoring="r2")

In [5]:
# Print results
# should be <=0, since there is no usable information in X
print(f"Regression - Leaky mean R^2: {bad_scores.mean():.3f}")
print(f"Regression - Pipeline R^2:   {good_scores.mean():.3f}")

Regression - Leaky mean R^2: 0.564
Regression - Pipeline R^2:   -0.433


### Classification setup

As above, we fit a model to a pure noise dataset.

Here, the target `y` is binary with an equal probability of being 0 and 1.

Hence, any model should be random guessing, yielding an accuracy of around 50%.

Implementing cross-validation incorrectly will again yield an overly optimistic result.

In [6]:
# Configurable params
k_best = 10
estimator = LogisticRegression(max_iter=1000)
rng = np.random # .RandomState(0) # for reproducibility

# Data setup
X = rng.normal(size=(200, 10000))    # random features
y = rng.randint(0, 2, size=200)    # random binary labels

# Cross-validation setup
cv = KFold(n_splits=10, shuffle=True, random_state=0)

In [7]:
# BAD: feature selection fit on full data (leakage!)
X_selected = SelectKBest(f_classif, k=k_best).fit_transform(X, y)
bad_scores = cross_val_score(estimator, X_selected, y, cv=cv, scoring="accuracy")

In [8]:
# GOOD: feature selection inside a Pipeline
pipe = Pipeline([
    ("select", SelectKBest(f_classif, k=k_best)),
    ("est", estimator),
])
good_scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

In [9]:

# Print results
# Should be around 0.5 since there is no usable information in X
print(f"Classification - Leaky mean acc: {bad_scores.mean():.3f}")
print(f"Classification - Pipeline acc:   {good_scores.mean():.3f}")

Classification - Leaky mean acc: 0.810
Classification - Pipeline acc:   0.525
